<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


# Build and Discover a ROS 2 Package

The previous notebook located the publisher and subscriber source files in their package and traced their executable names to Python functions. Those installation instructions now need to be applied so that ROS 2 can find the package and its executables.

After changing a publisher or subscriber, the terminal needs to find the updated package. A workspace build prepares its installed files, and sourcing the workspace overlay makes them discoverable to ROS 2. 

This notebook builds `ros2_pubsub`, loads the resulting workspace environment, and inspects the installed package. The observations will connect three steps: discovering source, preparing an installation, and making that installation available to ROS 2 commands.

## Prepare the terminal

The commands run in one terminal inside the dedicated environment described in the [README](../README.md#prerequisites), starting from the LX checkout used in the previous notebook.

Load ROS 2, save the checkout’s location, and move to the workspace root:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
LX_DIR="$PWD"
cd "$COLCON_WS"
```

Successful execution normally produces no output. `LX_DIR` preserves the checkout path, so that source files can still be located after changing directories. In the prepared editor, the checkout path is `/code/src/lx-dd-middleware-ros2` and the workspace root is `/code`.

The familiar source-discovery command can now use that saved path:

```bash
colcon list --base-paths "$LX_DIR/packages"
```

As previously observed, in the editor environment, the package appears as:

```text
ros2_pubsub	/code/src/lx-dd-middleware-ros2/packages/ros2_pubsub	(ros.ament_python)
```

The path (the middle output) identifies the source that will be built. Keeping discovery scoped to this checkout also avoids accidentally including another copy of the package elsewhere in the environment. Before building, it is good practice to check whether the terminal already knows of an installed version:

```bash
ros2 pkg prefix ros2_pubsub
```

The expected response at first execution of this notebook is:

```text
Package not found
```

That result is expected before the package has been installed and made available to the terminal. If a path appears instead, an existing installation is already visible; its location can be compared with the result after the build.

## Build the supplied package

The build uses the installation instructions inspected in the previous notebook to prepare the Python code, package metadata, and executable entry points.

From the workspace root:

```bash
colcon build --base-paths "$LX_DIR/packages" \
  --packages-select ros2_pubsub
```

`--base-paths` identifies where source discovery begins, while `--packages-select` chooses the package to build from those discovered there.

The build should finish with output such as:

```text
Starting >>> ros2_pubsub
--- stderr: ros2_pubsub
warning: build_py: byte-compiling is disabled, skipping.

warning: install_lib: byte-compiling is disabled, skipping.

---
Finished <<< ros2_pubsub [0.98s]

Summary: 1 package finished [1.10s]
  1 package had stderr output: ros2_pubsub
```

The `Finished` line and summary report that the package completed successfully. Note that the execution times vary based on the machine used. 

In this environment, Python byte-compilation is disabled, which accounts for the two warnings and the summary’s mention of stderr output. These warnings mean that the installation step skipped generating cached Python bytecode (`.pyc` files). The installed Python source can still run normally, so these particular warnings can be ignored for this LX when the build completes successfully and the package-discovery checks below pass. 

The build produces its working files under `build`, its installed files under `install`, and its diagnostic records under `log`. For this package, the installation is placed in:

```text
/code/install/ros2_pubsub
```

Application changes belong in the source checkout, where they can be incorporated by a subsequent build.

## Make the installation available to the terminal

Although a completed build leaves files on disk, the current shell also needs the search paths that allow ROS 2 to locate them.

If no installation was already visible, repeating the package lookup after the first build and before loading the overlay should still report that the package cannot be found:

```bash
ros2 pkg prefix ros2_pubsub
```

```text
Package not found
```

The workspace provides a setup script that adds its installed packages to the current shell’s environment (successful sourcing normally produces no output):

```bash
source install/setup.bash
```

The installed ROS 2 distribution supplies the underlying libraries and tools, while the workspace adds the exercise package on top. In this arrangement, the distribution is the **underlay** and the workspace is an **overlay**.

Sourcing applies the overlay’s environment settings to the shell executing the command, allowing subsequent ROS 2 commands in that shell to find its packages.


## Confirm package discovery

With the overlay loaded, the same lookup should resolve the installed package. That is, now:

```bash
ros2 pkg prefix ros2_pubsub
```

should find:

```text
/code/install/ros2_pubsub
```

This is the installation prepared by the build. Checking the prefix is particularly useful when several workspaces contain a package with the same name, because it identifies the installation selected by the current terminal.

The next check connects that installation to the executable registrations examined in notebook 5:

```bash
ros2 pkg executables ros2_pubsub
```

Both supplied executables should appear:

```text
ros2_pubsub my_publisher
ros2_pubsub my_subscriber
```

Each line contains a package name followed by an executable name. These entries provide the names needed by `ros2 run`, whose use is introduced in the next notebook.


## Use the package from another terminal

Each terminal has its own ROS 2 environment, so a terminal opened later also needs access to the distribution and workspace installation.

Open a new terminal and run:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
source "$COLCON_WS/install/setup.bash"
ros2 pkg prefix ros2_pubsub
```

The sourcing commands normally remain silent, followed by:

```text
/code/install/ros2_pubsub
```

The editor may already load these setup files automatically when opening a terminal. The explicit commands establish the environment needed by the activity and also apply to the alternate runtime shell described in the README.

A newly opened terminal can use the existing installation without rebuilding it. Rebuilding becomes necessary when source or package configuration changes need to be incorporated into the installed version.

## Recover package discovery

If `ros2 run` cannot find the package, check the build and current terminal's overlay in this order. In a new terminal, first load ROS 2 and set `LX_DIR="$PWD"` from the LX checkout, as in “Prepare the terminal”:

1. Return to `$COLCON_WS`.

2. Build `ros2_pubsub` with `colcon build --base-paths "$LX_DIR/packages" --packages-select ros2_pubsub`.

3. Source `install/setup.bash` in the terminal that will run the command.

4. Confirm discovery with `ros2 pkg prefix ros2_pubsub`.

A build error points to package preparation; a successful build followed by discovery after sourcing points to the terminal's environment. This order helps locate the problem before changing node code.
 
## Further reading

The [colcon quick-start guide](https://colcon.readthedocs.io/en/released/user/quick-start.html) describes package selection, workspace builds, and loading the resulting environment.

## Checkpoint

The next cell contains the interactive self-check. A response must be entered or selected before the corresponding answer can be revealed.

## Summary and next steps

Building applied the package’s installation instructions, and sourcing the overlay made the resulting installation available to the terminal. Package-prefix and executable queries then identified the selected installation and its registered programs.

The package is ready for an execution check. [Notebook 7: Inspect a Local ROS 2 Graph](./7-inspect-a-local-ros-2-graph.ipynb) starts the publisher and connects its running node to its topic, message rate, and received data.

In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
